**TIP:** You can save your work offline by using `File -> Download`.
Later you can upload the notebook back into the workspace.

## Setup

In [ ]:
# SETUP CELL
%pip -q install numpy sympy matplotlib ipywidgets pandas plotly anywidget scipy

In [ ]:
# SETUP CELL
import src_path
from gu_toolkit import *
from helpers.Fourier_02_helper import *

# Fourier coefficients of the square wave via `NIntegrate`

In `Fourier_02` you adjusted sine coefficients by hand.
In this notebook you will first **compute** those coefficients numerically, and then use them to drive the sliders yourself.

We work with the 1-periodic square wave
$$
\mathrm{Sq}(x)=
\begin{cases}
-1 &\text{if } x<0,\\
1 &\text{if } x>0.
\end{cases}
$$
Because this function is odd, only sine terms appear in its Fourier series.

## The square wave

In [ ]:
@NamedFunction
def Sq(x):
    return sign(sin(2 * pi * x))

fig_square = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-2, 2))
fig_square.title = r"The square wave $\mathrm{Sq}(x)$"
display(fig_square)

with fig_square:
    plot(Sq(x), x, id="Sq", label=r"$\mathrm{Sq}(x)$")

## Compute the Fourier coefficients

For the square wave, the sine coefficients are
$$
c_n = 2\int_{-1/2}^{1/2} \mathrm{Sq}(x)\sin(2\pi n x)\,dx.
$$
The next cell computes these numbers with `NIntegrate`.

In [ ]:
Nmax = 100

FTb = np.zeros(Nmax + 1)
for n in range(1, Nmax + 1):
    FTb[n] = NIntegrate(
        2 * Sq(x) * sin(2 * pi * n * x),
        (x, -1 / 2, 1 / 2),
    )



coeff_table = pd.DataFrame(
    {
        "n": np.arange(1, 9 + 1),
        "target value for b_n": FTb[1:10],
        "|b_n|": np.abs(FTb[1:10]),
    }
)
display(coeff_table.round(3))

## Part 1. Use the table to set the slider parameters

Use the values from the table as the target coefficients in
$$
M(x)=a_1\sin(2\pi x)+a_2\sin(2\pi 2x)+\cdots+a_{N_{\max}}\sin(2\pi N_{\max}x).
$$
Start with all sliders at `0`. Then copy the computed values from the table into the sliders `a_1, a_2, \ldots, a_{N_{\max}}`.

In [ ]:
slider_limit = 2.0
a_symbols = [a[n] for n in range(1, Nmax + 1)]

model = 0
for n in range(1, 9 + 1):
    model += a[n] * sin(2 * pi * n * x)

fig = Figure(x_range=(-1 / 2, 1 / 2))
fig.show()

with fig:
    set_title(r"Square wave and the slider-controlled sine model")
    plot(Sq(x), x, id="Sq(x)", label=r"$\mathrm{Sq}(x)$")
    plot(model, x, id="model", label=r"$M(x)$")
    # Set the default ranges for the parameters
    for a_parameter in [a[n] for n in range(1, 9 + 1)]:
        parameter(a_parameter, min=-slider_limit, max=slider_limit, step=0.001, value=0.0)

## Part 2. Add one parameter-controlled partial sum

Instead of plotting many different partial sums, build a **single** expression
$$
S_N(x)=\sum_{n=1}^{N_{\max}} b_n \cdot \sin(2\pi n x)\cdot \left(\begin{cases}1& \text{ if }n\leq N\\ 0 &\text{ otherwise}\end{cases}\right).
$$
Then use the slider `N` to decide how many terms are active.

After you have entered the coefficients from the table, move the `N` slider between `0` and `Nmax`.

In [ ]:
N = symbols("N")

S_N = 0
for n in range(1, Nmax + 1):
    S_N += FTb[n] * Piecewise((1, n <= N), (0, True)) * sin(2 * pi * n * x)

with fig:
    plot(S_N, x, id="S_N", label=r"$S_N(x)$")
    parameter(N, min=0, max=Nmax, step=1, value=0)

fig.info(MaxDistanceCard(x, Sq(x), S_N), id="sq_minus_SN_max")
fig.info(AvgDistanceCard(x, Sq(x), S_N), id="sq_minus_SN_avg")

## Explore

- Which coefficients are essentially `0`?
- What do you notice when you move `N` through the even values?
- After you enter the table values, how does `S_N` behave near the jump at `x=0`?
- Compare the numerical values with the exact formula
  $$
  c_n = \frac{4}{\pi n}\text{ for odd }n,\qquad c_n = 0\text{ for even }n.
  $$